# 11 — Prediction Storage ke MongoDB & Export untuk Power BI

**Tujuan notebook ini:**
- **Fase 11**: Simpan hasil scoring ke MongoDB collection `customer_predictions` — MongoDB sekarang punya **RAW LAYER + PREDICTION LAYER**
- **Fase 12 (persiapan)**: Konsolidasi & validasi semua tabel processed yang siap diimport ke Power BI (Opsi A — via CSV di `data_processed/`)


In [1]:
import pandas as pd
from pymongo import MongoClient
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data_processed")
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "olist_db"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

customer_predictions = pd.read_csv(DATA_DIR / "customer_predictions.csv")
print(f"customer_predictions loaded: {customer_predictions.shape}")
customer_predictions.head()


customer_predictions loaded: (54738, 7)


,customer_unique_id,repeat_purchase_probability,prediction,threshold_used,threshold_method,customer_value_observation,value_band
0,3e43e6105506432c953e165fb2acf44c,0.998548,1,0.5987,top_5_percent,1172.66,High Value
1,8d50f5eadf50201ccdcedfb9e2ac8455,0.998318,1,0.5987,top_5_percent,503.54,High Value
2,1b6c7548a2a1f9037c1fd3ddfed95f33,0.993663,1,0.5987,top_5_percent,959.01,High Value
3,47c1a3033b8b77b3ab6e109eb4d5fdf3,0.987175,1,0.5987,top_5_percent,944.21,High Value
4,6469f99c1f9dfae7733b25662e7f1782,0.985811,1,0.5987,top_5_percent,726.18,High Value


---
## Fase 11 — Prediction Storage ke MongoDB

Buat collection `customer_predictions`. Sesuai roadmap, tiap record menyimpan `threshold_used` dan `threshold_method` — supaya `prediction` (label biner) bisa dijelaskan asal-usulnya, bukan sekadar angka yang muncul begitu saja.


In [2]:
SCORING_DATE = datetime.now().strftime("%Y-%m-%d")
MODEL_NAME = "LogisticRegression"

records = customer_predictions.copy()
records["model"] = MODEL_NAME
records["scoring_date"] = SCORING_DATE

records_dict = records.to_dict(orient="records")

# Drop & recreate collection supaya notebook ini idempotent (bisa di-run ulang tanpa duplikat)
db["customer_predictions"].drop()
db["customer_predictions"].insert_many(records_dict)

mongo_doc_count = db["customer_predictions"].count_documents({})
print(f"CSV rows       : {len(customer_predictions):,}")
print(f"MongoDB docs   : {mongo_doc_count:,}")
print(f"Match          : {len(customer_predictions) == mongo_doc_count}")


CSV rows       : 54,738
MongoDB docs   : 54,738
Match          : True


In [3]:
print("Contoh 1 dokumen dari customer_predictions:")
print(db["customer_predictions"].find_one())


Contoh 1 dokumen dari customer_predictions:
{'_id': ObjectId('6a993b957ba2af8c5e2751c5'), 'customer_unique_id': '3e43e6105506432c953e165fb2acf44c', 'repeat_purchase_probability': 0.9985477121745476, 'prediction': 1, 'threshold_used': 0.5987, 'threshold_method': 'top_5_percent', 'customer_value_observation': 1172.66, 'value_band': 'High Value', 'model': 'LogisticRegression', 'scoring_date': '2026-09-03'}


---
## Validasi: MongoDB Sekarang Punya Dua Layer


In [4]:
all_collections = db.list_collection_names()
raw_collections = [c for c in all_collections if c.endswith("_raw")]
prediction_collections = [c for c in all_collections if c == "customer_predictions"]

print("RAW LAYER:")
for c in raw_collections:
    print(f"  - {c}: {db[c].count_documents({}):,} docs")

print("\nPREDICTION LAYER:")
for c in prediction_collections:
    print(f"  - {c}: {db[c].count_documents({}):,} docs")

print(f"\nTotal collections di olist_db: {len(all_collections)}")


RAW LAYER:
  - category_translation_raw: 71 docs
  - sellers_raw: 3,095 docs
  - geolocation_raw: 1,000,163 docs
  - orders_raw: 99,441 docs
  - payments_raw: 103,886 docs
  - products_raw: 32,951 docs
  - customers_raw: 99,441 docs
  - reviews_raw: 99,224 docs
  - order_items_raw: 112,650 docs

PREDICTION LAYER:
  - customer_predictions: 54,738 docs

Total collections di olist_db: 10


---
## Fase 12 (Persiapan) — Konsolidasi Tabel untuk Power BI

Sesuai roadmap: **"Jangan tarik raw MongoDB mentah untuk semua visual."** Power BI ambil dari `data_processed/` (Opsi A — via CSV), bukan langsung dari raw collection.


In [5]:
expected_files = [
    "customer_analytics.csv",       # Dashboard 1, 2 - business analytics (full-history)
    "product_analytics.csv",        # Dashboard 3
    "seller_analytics.csv",         # Dashboard 3, 4
    "customer_segmentation_full.csv",  # Dashboard 2 - segmentation
    "customer_predictions.csv",     # Dashboard 5 - predictive analytics
]

print("Cek kelengkapan file untuk Power BI:")
for filename in expected_files:
    filepath = DATA_DIR / filename
    if filepath.exists():
        df_check = pd.read_csv(filepath)
        print(f"  ✅ {filename}: {df_check.shape[0]:,} rows, {df_check.shape[1]} columns")
    else:
        print(f"  ⚠️  {filename}: TIDAK DITEMUKAN")


Cek kelengkapan file untuk Power BI:
  ✅ customer_analytics.csv: 96,096 rows, 8 columns
  ✅ product_analytics.csv: 32,951 rows, 7 columns
  ✅ seller_analytics.csv: 3,095 rows, 6 columns
  ✅ customer_segmentation_full.csv: 96,096 rows, 7 columns
  ✅ customer_predictions.csv: 54,738 rows, 7 columns


---
## Tabel Konsolidasi Khusus — Dashboard 5 (Predictive Analytics / Priority Matrix)

Gabungkan `customer_predictions` dengan `customer_segmentation_full` (dari Fase 6) supaya Power BI tinggal 1 tabel untuk Priority Matrix, bukan join manual di Power BI.


In [6]:
customer_segmentation = pd.read_csv(DATA_DIR / "customer_segmentation_full.csv")

dashboard5_table = customer_predictions.merge(
    customer_segmentation[["customer_unique_id", "segment"]],
    on="customer_unique_id", how="left"
)

# Priority Matrix quadrant label (gabungan business segment full-history + ML prediction observation)
def assign_quadrant(row):
    high_value = row["value_band"] == "High Value"
    high_prob = row["prediction"] == 1
    if high_value and high_prob:
        return "VIP / RETAIN"
    elif high_value and not high_prob:
        return "WIN-BACK"
    elif not high_value and high_prob:
        return "GROW"
    else:
        return "LOW PRIORITY"

dashboard5_table["priority_quadrant"] = dashboard5_table.apply(assign_quadrant, axis=1)

print("Distribusi Priority Quadrant:")
print(dashboard5_table["priority_quadrant"].value_counts())

dashboard5_table.to_csv(DATA_DIR / "dashboard5_predictive_analytics.csv", index=False)
print(f"\nTersimpan: {DATA_DIR / 'dashboard5_predictive_analytics.csv'}")


Distribusi Priority Quadrant:
priority_quadrant
LOW PRIORITY    26098
WIN-BACK        25903
VIP / RETAIN     1471
GROW             1266
Name: count, dtype: int64

Tersimpan: ..\data_processed\dashboard5_predictive_analytics.csv


> **Catatan:** `value_band` di tabel ini berdasarkan `customer_value_observation` (Fase 10), sedangkan `segment` (High/Medium/Low Value dari K-Means, Fase 6) berdasarkan `customer_analytics_full`. Keduanya boleh ditampilkan berdampingan di Power BI untuk konteks tambahan, tapi **`priority_quadrant`** (dipakai untuk Priority Matrix) murni dari `value_band` + `prediction`, konsisten information set observation.


---
## Fase 13 — Business Recommendations & Cost/ROI Analysis

**Pertanyaan bisnis:** kalau tim marketing pakai model ini untuk targeting campaign, seberapa besar penghematan biaya / peningkatan ROI dibanding random targeting?

> ⚠️ **Dataset Olist tidak menyediakan data biaya campaign riil** (cost per kontak, biaya customer service, dst). Angka biaya & value di bawah ini pakai **asumsi eksplisit** yang dinyatakan jelas — perusahaan yang beneran pakai analisis ini WAJIB mengganti asumsi dengan angka riil mereka sendiri. Tujuan bagian ini adalah menunjukkan **kerangka perhitungan**, bukan angka final yang bisa langsung dipakai.


In [7]:
# === ASUMSI EKSPLISIT (WAJIB diganti dengan angka riil perusahaan) ===
COST_PER_CONTACT = 5.0       # asumsi: biaya kirim email/SMS/notifikasi per customer
VALUE_PER_REPEAT_CUSTOMER = dashboard5_table["customer_value_observation"].mean()  # proxy: rata-rata transaction value historis
N_CONTACTED = int((dashboard5_table["prediction"] == 1).sum())  # jumlah customer yang di-flag (Top 5%)

print(f"Asumsi COST_PER_CONTACT           : {COST_PER_CONTACT} (unit mata uang, ilustratif)")
print(f"Asumsi VALUE_PER_REPEAT_CUSTOMER  : {VALUE_PER_REPEAT_CUSTOMER:,.2f} (proxy dari avg customer_value_observation)")
print(f"N_CONTACTED (Top 5%, sama utk kedua skenario): {N_CONTACTED:,}")


Asumsi COST_PER_CONTACT           : 5.0 (unit mata uang, ilustratif)
Asumsi VALUE_PER_REPEAT_CUSTOMER  : 163.08 (proxy dari avg customer_value_observation)
N_CONTACTED (Top 5%, sama utk kedua skenario): 2,737


In [8]:
# Baseline positive rate (random targeting) & precision di threshold (model-based targeting)
# Angka ini diambil dari hasil aktual notebook 07 (Decision Gate) & notebook 10 (scoring) - bukan asumsi
baseline_rate = ml_dataset["target"].mean() if "target" in dir() else customer_predictions.merge(
    pd.read_csv(DATA_DIR / "ml_dataset_observation.csv")[["customer_unique_id", "target"]],
    on="customer_unique_id", how="left"
)["target"].mean()
precision_at_threshold = customer_predictions.loc[
    customer_predictions["prediction"] == 1
].merge(
    pd.read_csv(DATA_DIR / "ml_dataset_observation.csv")[["customer_unique_id", "target"]],
    on="customer_unique_id", how="left"
)["target"].mean()

print(f"Baseline conversion rate (random)         : {baseline_rate*100:.2f}%")
print(f"Precision @ Top 5% (model-based)          : {precision_at_threshold*100:.2f}%")

scenarios = pd.DataFrame([
    {
        "scenario": "Random Targeting (tanpa model)",
        "n_contacted": N_CONTACTED,
        "conversion_rate": baseline_rate,
    },
    {
        "scenario": "Model-based Targeting (Top 5%)",
        "n_contacted": N_CONTACTED,
        "conversion_rate": precision_at_threshold,
    },
])

scenarios["expected_conversions"] = scenarios["n_contacted"] * scenarios["conversion_rate"]
scenarios["total_cost"] = scenarios["n_contacted"] * COST_PER_CONTACT
scenarios["expected_revenue"] = scenarios["expected_conversions"] * VALUE_PER_REPEAT_CUSTOMER
scenarios["net_benefit"] = scenarios["expected_revenue"] - scenarios["total_cost"]
scenarios["cost_per_acquisition"] = scenarios["total_cost"] / scenarios["expected_conversions"]

print("Perbandingan Skenario (dengan asumsi di atas):")
print(scenarios.to_string(index=False))


Baseline conversion rate (random)         : 1.21%
Precision @ Top 5% (model-based)          : 3.32%
Perbandingan Skenario (dengan asumsi di atas):
                      scenario  n_contacted  conversion_rate  expected_conversions  total_cost  expected_revenue  net_benefit  cost_per_acquisition
Random Targeting (tanpa model)         2737         0.012131             33.201213     13685.0       5414.541871 -8270.458129            412.183735
Model-based Targeting (Top 5%)         2737         0.033248             91.000000     13685.0      14840.521323  1155.521323            150.384615


In [9]:
uplift_conversions = scenarios.loc[1, "expected_conversions"] - scenarios.loc[0, "expected_conversions"]
uplift_net_benefit = scenarios.loc[1, "net_benefit"] - scenarios.loc[0, "net_benefit"]
cpa_reduction_pct = (
    (scenarios.loc[0, "cost_per_acquisition"] - scenarios.loc[1, "cost_per_acquisition"])
    / scenarios.loc[0, "cost_per_acquisition"] * 100
)

print(f"Selisih expected conversions (Model vs Random) : +{uplift_conversions:.1f} customer")
print(f"Selisih net benefit (Model vs Random)          : {uplift_net_benefit:+,.2f}")
print(f"Penurunan cost per acquisition                  : {cpa_reduction_pct:.1f}%")

print(f"\n📌 Dengan {N_CONTACTED:,} customer yang sama-sama dikontak, targeting berbasis model")
print(f"   menghasilkan ~{uplift_conversions:.0f} konversi lebih banyak dibanding random targeting,")
print(f"   dengan cost per acquisition {cpa_reduction_pct:.0f}% lebih murah — TANPA menambah budget kontak.")


Selisih expected conversions (Model vs Random) : +57.8 customer
Selisih net benefit (Model vs Random)          : +9,425.98
Penurunan cost per acquisition                  : 63.5%

📌 Dengan 2,737 customer yang sama-sama dikontak, targeting berbasis model
   menghasilkan ~58 konversi lebih banyak dibanding random targeting,
   dengan cost per acquisition 64% lebih murah — TANPA menambah budget kontak.


**Catatan interpretasi:**
- Kerangka ini menunjukkan **arah dan mekanisme** manfaat model (lebih banyak konversi & cost per acquisition lebih rendah untuk jumlah kontak yang sama), bukan angka ROI final yang presisi.
- `COST_PER_CONTACT` dan `VALUE_PER_REPEAT_CUSTOMER` di sini ilustratif — kalau perusahaan punya data biaya campaign riil (cost email/SMS/call center) dan margin/profit riil (bukan cuma transaction value), angka ini harus diganti untuk keputusan bisnis yang akurat.
- Analisis ini mengasumsikan model dipakai sekali (single campaign). Kalau dipakai berulang (recurring scoring tiap bulan/kuartal), manfaat kumulatifnya bisa lebih besar — tapi juga perlu mempertimbangkan model drift (performa bisa menurun seiring waktu, perlu re-training berkala).


### Rekomendasi Bisnis per Kuadran (Priority Matrix)

| Kuadran | Populasi | Rekomendasi |
|---|---|---|
| VIP / RETAIN (High Value + High Probability) | lihat `dashboard5_table` | Maintain / loyalty program — customer bernilai tinggi yang memang cenderung repeat |
| WIN-BACK (High Value + Low Probability) | lihat `dashboard5_table` | Campaign win-back terfokus — customer bernilai tinggi tapi probability repeat rendah, potensi kerugian terbesar kalau churn |
| GROW (Low Value + High Probability) | lihat `dashboard5_table` | Cross-sell / upsell — customer yang cenderung repeat tapi nilai transaksinya masih kecil |
| LOW PRIORITY (Low Value + Low Probability) | lihat `dashboard5_table` | Low-cost marketing / automated saja, bukan fokus utama campaign personal |


---
## Definition of Done (Fase 11 & Persiapan Fase 12)

- [ ] Collection `customer_predictions` tersimpan di MongoDB, row count = document count
- [ ] MongoDB terkonfirmasi punya 2 layer: RAW LAYER + PREDICTION LAYER
- [ ] Tiap record prediksi punya `threshold_used`, `threshold_method`, `model`, `scoring_date` — bisa dipertanggungjawabkan
- [ ] Semua 5 file processed tersedia lengkap untuk Power BI (Opsi A — via CSV)
- [ ] Tabel konsolidasi `dashboard5_predictive_analytics.csv` siap untuk Priority Matrix

**Lanjut ke:** Power BI (Fase 12) — import semua CSV dari `data_processed/`, susun 5 dashboard sesuai roadmap. Setelah itu: Fase 13 (Business Recommendations) dan Fase 14 (Documentation).
